# 00.5 — The command line for MATLAB users

Everything in this project — creating the venv, installing packages, running tests, launching training, submitting SLURM jobs — happens in a **terminal**. If you've lived inside the MATLAB desktop, the terminal may be the single most unfamiliar tool in this curriculum, and it's a prerequisite for all the others.

This notebook covers:

* What a shell is and how it relates to the terminal window.
* Navigating: `pwd`, `ls`, `cd`, absolute vs relative paths, `~`.
* Running programs, flags, and arguments.
* Environment variables — including this project's `NDD_MATLAB_SOURCE_ROOT`.
* `PATH` and how the shell finds programs.
* `source` vs running a script (why it's `source .venv/bin/activate`).
* Shell config files (`.zshrc` / `.bashrc`).

**No prerequisites** — you can read this before anything else in the curriculum.

## Section 1 — What MATLAB does

MATLAB's Command Window IS a shell — just one whose language is MATLAB:

```matlab
pwd                      % print working directory
cd('~/Documents')        % change directory
ls                       % list files (calls the OS)
!echo hello              % the ! prefix escapes to the system shell
setenv('FOO', 'bar')     % set an environment variable
getenv('FOO')            % read one
```

If you've used `!` in MATLAB, you've already used the system shell. The difference now: the shell becomes your primary tool instead of an escape hatch, and its language (`bash` or `zsh`) has its own tiny syntax.

Your `startup.m` has a shell analog too — `.zshrc` (macOS default) or `.bashrc` (most Linux), which runs every time a new shell starts.

## Section 2 — The Python concepts you need

### 2.1 — Terminal vs shell

The **terminal** is the window; the **shell** is the program running inside it that reads your commands. macOS ships `zsh`; most Linux distros ship `bash`. For everything in this curriculum they behave identically.

A note about running shell commands FROM a notebook (which we'll do below): prefixing a cell line with `!` hands it to the shell — the same trick as MATLAB's `!`. But **each `!` line runs in a fresh subshell**, so `!cd somewhere` does NOT persist to the next line. Real terminal sessions keep state; notebook `!` lines don't. Keep that in mind as you run the demos.

In [ ]:
# The `!` prefix works here just like MATLAB's `!` — try the basics:
!pwd

In [ ]:
!ls ../..

### 2.2 — Navigation and paths

| MATLAB | Shell | Meaning |
|---|---|---|
| `pwd` | `pwd` | print working directory |
| `cd('dir')` | `cd dir` | change directory (no quotes/parens) |
| `ls` / `dir` | `ls` (`ls -la` for detail) | list files |
| `mkdir('d')` | `mkdir d` (`mkdir -p a/b/c` for nested) | make directory |
| `delete('f')` | `rm f` (`rm -r d` for directories — careful!) | remove |
| `copyfile(a,b)` | `cp a b` | copy |
| `movefile(a,b)` | `mv a b` | move / rename |

**Path anatomy:**

* `/Users/you/project/data.mat` — **absolute** path (starts with `/`).
* `data/trial.mat` — **relative** to the current directory.
* `..` — parent directory; `.` — current directory.
* `~` — your home directory (`/Users/you` on macOS, `/home/you` on Linux).
* **Spaces in paths need quotes**: `cd "Neural Data Reading"` — this is why Python projects avoid spaces in directory names.

Tab completion works everywhere: type the first few letters and press Tab. It's the single biggest speed multiplier in a terminal — use it constantly, and it also prevents typos in long paths.

### 2.3 — Running programs, flags, arguments

Shell command anatomy — the same shape you've seen throughout this curriculum:

```bash
python -m pytest tests/unit/test_mat_files.py -q
└─┬──┘ └──────┬─────┘ └────────────┬─────────┘ └┬┘
program   flag+value        argument          flag
```

* **Flags** start with `-` (short, stackable: `ls -la`) or `--` (long: `--config-name`).
* Flags may take values: `--fold 3` or `--fold=3` (both forms work in most tools).
* `--help` is near-universal — every tool in this project answers it:

In [ ]:
import sys
# {sys.executable} interpolates the kernel's own Python into the shell line —
# no activation needed, because this notebook's kernel IS the venv's Python.
!{sys.executable} -m neural_data_decoding --help

(In a real terminal you'd `source .venv/bin/activate` once and then plain `python -m ...` works. Inside a notebook, the trick above interpolates the kernel's own interpreter path into the shell line with `{sys.executable}` — IPython substitutes Python variables into `!` lines wrapped in braces. You'll also see the `&&` operator chaining commands: "run the next only if the previous succeeded.")

### 2.4 — Environment variables

Environment variables are named strings that live in the shell and are inherited by every program the shell launches. MATLAB's `getenv`/`setenv`, exactly.

```bash
export NDD_MATLAB_SOURCE_ROOT="/Users/you/Documents/MATLAB/Neural Data Reading"
echo $NDD_MATLAB_SOURCE_ROOT     # read it back — note the $ prefix when READING
```

**This project uses exactly this mechanism**: the parity tests find the MATLAB sources through `NDD_MATLAB_SOURCE_ROOT` (see `CLAUDE.md`). An `export` in a terminal lasts only until that terminal closes — to make it permanent, add the `export` line to your shell config file (Section 2.6).

From Python, environment variables surface via `os.environ` (taught in [01.8](../01_python_for_matlab_users/01.8_the_python_standard_library_for_matlab_users.ipynb)):

In [ ]:
import os
print(os.environ.get("NDD_MATLAB_SOURCE_ROOT", "<not set in this kernel's environment>"))
print(os.environ.get("HOME"))

### 2.5 — `PATH`: how the shell finds programs

When you type `python`, the shell searches the directories listed in the `PATH` environment variable, in order, and runs the **first** match. This is the same idea as the MATLAB path — but for programs instead of `.m` files.

`which python3` tells you which one won:

In [ ]:
!echo "first PATH entries:" && echo $PATH | tr ':' '\n' | head -6
!echo "---" && which python3

**This is the mechanism behind venv activation.** `source .venv/bin/activate` simply prepends `<repo>/.venv/bin` to `PATH` — so `python` and `pip` now resolve to the venv's copies instead of the system ones. Deactivating removes that entry. No magic, just `PATH` order.

That's also why the wrong-kernel problems in [00.4](00.4_ide_deep_dive.ipynb) all reduce to one question: *which `python` is first on the `PATH` (or picked by the editor)?*

### 2.6 — `source` vs running, and shell config files

Two ways to execute a script file of shell commands:

* `bash script.sh` — runs it in a **child** shell. Variables it sets vanish when it exits.
* `source script.sh` — runs it in **your current** shell. Variables it sets stick around.

`activate` must change YOUR shell's `PATH`, hence `source .venv/bin/activate` — running it as a child would change the child's `PATH` and then discard it.

**Shell config files** are `source`d automatically when a shell starts:

| Shell | File | MATLAB analog |
|---|---|---|
| zsh (macOS default) | `~/.zshrc` | `startup.m` |
| bash (most Linux) | `~/.bashrc` | `startup.m` |

Put persistent `export`s there — e.g. this project's `NDD_MATLAB_SOURCE_ROOT`. After editing, either open a new terminal or `source ~/.zshrc` to apply.

## Section 3 — The `neural_data_decoding` implementation

Every shell command the project documentation asks you to run, decoded:

| Command | What each piece does |
|---|---|
| `source .venv/bin/activate` | modify current shell's PATH so python/pip resolve to the venv |
| `pip install -e ".[dev]"` | pip = the venv's pip (PATH); quotes protect the brackets from zsh's glob expansion |
| `python -m pytest -q` | run module `pytest` under the venv's python; `-q` flag = quiet |
| `python -m neural_data_decoding train --config-name real_data_base --fold 1` | the project CLI: subcommand + two long flags with values |
| `jupyter lab notebooks/` | program installed by pip into `.venv/bin` (on PATH after activation) + one relative-path argument |
| `export NDD_MATLAB_SOURCE_ROOT="..."` | env var the parity fixtures read; quotes because the path contains spaces |

The `"[dev]"` quoting detail is worth pinning: in zsh, unquoted square brackets trigger filename matching, so `pip install -e .[dev]` can fail with `no matches found` — the quotes prevent that.

## Section 4 — Hands-on exercises

### Exercise 4.1 — navigate and inspect

In a REAL terminal (not this notebook): `cd` to the repo, list the `configs/` directory, then print the first 10 lines of `configs/base.yaml` with `head`. Solution below.

In [ ]:
# Reveal (as run from a notebook; in a terminal drop the leading ! and the cd chaining):
!cd ../.. && ls configs && echo "---" && head -10 configs/base.yaml

### Exercise 4.2 — prove the fresh-subshell rule

Predict the output of the next cell before running it.

In [ ]:
!cd /tmp
!pwd

The `cd` ran in one subshell and died with it; `pwd` ran in a new subshell that started back in the notebook's directory. This is THE thing to remember about `!` cells.

## Section 5 — Common errors

### `command not found: python`

The program isn't on the `PATH`. Either the venv isn't activated (for venv-installed tools), or the program isn't installed. `which <name>` and `echo $PATH` are the diagnostics.

### `no matches found: .[dev]`

zsh tried to glob the square brackets. Quote it: `pip install -e ".[dev]"`.

### `No such file or directory`

The path is wrong relative to where you ARE (`pwd` first), or a space in the path broke the argument in two (quote it).

### `Permission denied`

You're writing somewhere you don't own (system directories) — work under `~` or the repo — or a script isn't executable (`bash script.sh` sidesteps that).

### Terminal changes don't persist

`export`s and `cd` last only for that terminal session. Persistent settings go in `~/.zshrc` / `~/.bashrc`. And in notebooks, they last only one `!` line.

### `source: command not found` (on Windows)

`source` is a Unix-shell builtin. In PowerShell the venv activation is `.venv\Scripts\Activate.ps1`; in cmd.exe it's `.venv\Scripts\activate.bat`.

## Section 6 — Further reading

- [The Missing Semester (MIT)](https://missing.csail.mit.edu/2020/course-shell/) — the best short course on shell fundamentals.
- [Software Carpentry — the Unix shell](https://swcarpentry.github.io/shell-novice/) — written for scientists.
- [zsh vs bash differences](https://apple.stackexchange.com/questions/361870/what-are-the-practical-differences-between-bash-and-zsh) — mostly ignorable, occasionally (globbing!) not.

Next up: **[00.6 git and GitHub](00.6_git_and_github_for_matlab_users.ipynb)**.